In [ ]:
import geopandas as gpd
import osmnx as ox
import pandas as pd

sal= pd.read_csv('../pop_pred_final.csv')
phm=pd.read_csv('../../PHARMACIES_MASTER_FINAL.csv')

In [ ]:
salpoint=gpd.read_file('../../2011_census/2011_Census/2011_ea_sal_centroids_kzn_gp.dbf')
salpoly=gpd.read_file('../../2011_census/2011_Census/ea_sal_kzn_gp.dbf')

In [1105]:
salpoint.loc[salpoint["EA_CODE"] == 59914808].geometry.centroid.iloc[0].coords[0]

C:\Users\kalmanj\AppData\Local\Temp\ipykernel_36976\1239686221.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  salpoint.loc[salpoint["EA_CODE"] == 59914808].geometry.centroid.iloc[0].coords[0]


(30.97351551826473, -29.723625416320004)

In [1106]:
pd.reset_option('display.float_format')

In [661]:
# kzndrive=ox.load_graphml('../../networks/network_kwazulu_natal_drive.graphml')
# gauwalk=ox.load_graphml('../../networks/network_gauteng_walk.graphml')
# gaudrive=ox.load_graphml('../../networks/network_gauteng_drive.graphml')

In [998]:
pharm_gdf = gpd.GeoDataFrame(
    phm,
    geometry=gpd.points_from_xy(phm['lng'], phm['lat']),
    crs="EPSG:4326"  # WGS84
)
pharm_gdf = pharm_gdf.to_crs(salpoly.crs)

In [6]:
sal = sal.merge(
    salpoly[["EA_CODE", "PR_NAME", 'geometry']],
    on="EA_CODE",
    how="left"
)
sal = gpd.GeoDataFrame(sal, geometry="geometry")
sal = sal[sal["sal2023_est"] >= 100].copy()

C:\Users\kalmanj\AppData\Local\Temp\ipykernel_19396\4115006631.py:1: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  sal = sal.merge(


In [591]:
# def assign_catchment(row):
#     if row['EA_GTYPE'] == 'Urban':
#         return 0.5 * 1609.34
#     elif row['EA_GTYPE'] == 'Traditional':
#         return 5 * 1609.34
#     else:
#         return 15 *1609.34

# sal['catchment'] = sal.apply(assign_catchment, axis=1)
# pharm_with_sal['catchment'] =5 * 1609.34

In [1000]:
kznsal = sal[sal["PR_NAME"] == "KwaZulu-Natal"].copy()
gausal = sal[sal["PR_NAME"] == "Gauteng"].copy()

In [ ]:
kznpharm = gpd.sjoin(
    pharm_gdf,
    kznsal,
    predicate='within'
)

gaupharm = gpd.sjoin(
    pharm_gdf,
    gausal,
    predicate='within'
)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

# plot polygons
kznsal.plot(
    ax=ax,
    color='lightgrey',     # fill color
    edgecolor='gray',     # borders
    linewidth=0.1
)

# (optional) plot pharmacies on top
kznpharm.plot(
    ax=ax,
    color='blue',
    markersize=2,
    alpha=0.8
)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

# plot polygons
gausal.plot(
    ax=ax,
    color='lightgrey',     # fill color
    edgecolor='gray',     # borders
    linewidth=0.1
)

# (optional) plot pharmacies on top
gaupharm.plot(
    ax=ax,
    color='blue',
    markersize=2,
    alpha=0.8
)

In [ ]:
# Kwazulu Natal Walk Analysis 

In [ ]:
#kznwalk=ox.load_graphml('../../networks/network_kwazulu_natal_walk.graphml')
kznwalk = ox.project_graph(kznwalk, to_crs="EPSG:32735")

In [ ]:
import osmnx as ox
import networkx as nx
from tqdm import tqdm
import geopandas as gpd
from shapely.geometry import Point

kznsal = kznsal.set_geometry(salpoint["geometry"])

kznsal = kznsal.to_crs(kznwalk.graph["crs"])
kznpharm = kznpharm.to_crs(kznwalk.graph["crs"])

In [ ]:
# GRAPH TRUNCATION
(maxy, miny, maxx, minx) = kznsal.total_bounds 
bbox = (maxy, miny, maxx, minx)

kznwalk = ox.truncate.truncate_graph_bbox(kznwalk, bbox)


KeyboardInterrupt



In [ ]:
# NODE ASSIGNMENT
kznsal["node"] = ox.nearest_nodes(
    kznwalk,
    kznsal.geometry.x,
    kznsal.geometry.y
)

pharm_gdf = kznpharm.copy()
# pharm_gdf = pharm_gdf[
#     pharm_gdf.geometry.x.notna() &
#     pharm_gdf.geometry.y.notna()
# ]

pharm_gdf["node"] = ox.nearest_nodes(
    kznwalk,
    pharm_gdf.geometry.x,
    pharm_gdf.geometry.y
)

In [1007]:
def decay(d, beta=0.0001): 
    return np.exp(-beta * d)

In [1008]:
pop_dict = kznsal.set_index("node")["sal2023_est"].to_dict()

Rj_list = []

for idx, pharm in tqdm(pharm_gdf.iterrows(), total=len(pharm_gdf)):

    lengths = nx.single_source_dijkstra_path_length(
        kznwalk,
        pharm["node"],
        cutoff=3000,
        weight="length"
    )

    weighted_pop = sum(
        pop_dict.get(node, 0) * decay(dist)
        for node, dist in lengths.items()
              )

    Rj = 1 / (weighted_pop/1000) if weighted_pop > 0 else 0
    Rj_list.append(Rj)

pharm_gdf["Rj"] = Rj_list

pharm_dict = pharm_gdf.groupby("node")["Rj"].sum().to_dict()

Ai_list = []

for idx, pop in tqdm(kznsal.iterrows(), total=len(kznsal)):

    lengths = nx.single_source_dijkstra_path_length(
        kznwalk,
        pop["node"],
        cutoff=3000,
        weight="length"
    )

    Ai = sum(
        pharm_dict.get(node, 0) * decay(dist)
        for node, dist in lengths.items()
    )

    Ai_list.append(Ai)

kznsal["Ai_walk"] = Ai_list

100%|███████████████████████████████████████████████████████████████████████████| 14094/14094 [00:15<00:00, 885.99it/s]


In [1009]:
weighted_pop

np.float64(0.0)

In [1010]:
kznsal["Ai_walk"].describe()

count    14094.000000
mean         0.034477
std          0.447682
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         40.487631
Name: Ai_walk, dtype: float64

In [1011]:
kznsal["Ai_walk"].describe(percentiles=[.9,0.95, 0.98, 0.99, 0.999])

count    14094.000000
mean         0.034477
std          0.447682
min          0.000000
50%          0.000000
90%          0.000000
95%          0.139316
98%          0.376296
99%          0.699541
99.9%        3.235036
max         40.487631
Name: Ai_walk, dtype: float64

In [ ]:
# Kwazulu Natal Road Analysis 

In [ ]:
#kzndrive=ox.load_graphml('../../networks/network_kwazulu_natal_drive.graphml')

In [844]:
kzndrive = ox.project_graph(kzndrive, to_crs="EPSG:32735")
kznsal = kznsal.to_crs(kzndrive.graph["crs"])
kznpharm = kznpharm.to_crs(kzndrive.graph["crs"])

In [ ]:
# GRAPH TRUNCATION
(maxy, miny, maxx, minx) = kznsal.total_bounds 
bbox = (maxy, miny, maxx, minx)

kzndrive = ox.truncate.truncate_graph_bbox(kzndrive, bbox)

In [ ]:
# NODE ASSIGNMENT
kznsal["node"] = ox.nearest_nodes(
    kzndrive,
    kznsal.geometry.x,
    kznsal.geometry.y
)

pharm_gdf["node"] = ox.nearest_nodes(
    kzndrive,
    pharm_gdf.geometry.x,
    pharm_gdf.geometry.y
)

In [1013]:
pop_dict = kznsal.set_index("node")["sal2023_est"].to_dict()

Rj_list = []

for idx, pharm in tqdm(pharm_gdf.iterrows(), total=len(pharm_gdf)):

    lengths = nx.single_source_dijkstra_path_length(
        kzndrive,
        pharm["node"],
        cutoff=10000,
        weight="length"
    )

    weighted_pop = sum(
        pop_dict.get(node, 0) * decay(dist)
        for node, dist in lengths.items()
    )


    Rj = 1 / (weighted_pop/1000) if weighted_pop > 0 else 0
    Rj_list.append(Rj)

pharm_gdf["Rj"] = Rj_list

pharm_dict = pharm_gdf.groupby("node")["Rj"].sum().to_dict()

Ai_list = []

for idx, pop in tqdm(kznsal.iterrows(), total=len(kznsal)):

    lengths = nx.single_source_dijkstra_path_length(
        kzndrive,
        pop["node"],
        cutoff=10000,
        weight="length"
    )

    Ai = sum(
        pharm_dict.get(node, 0) * decay(dist)
        for node, dist in lengths.items()
    )

    Ai_list.append(Ai)

kznsal["Ai_drive"] = Ai_list

100%|███████████████████████████████████████████████████████████████████████████| 14094/14094 [00:41<00:00, 343.40it/s]


In [1014]:
weighted_pop

np.float64(0.0)

In [1015]:
kznsal["Ai_drive"].describe()

count    14094.000000
mean         0.054672
std          0.576564
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         32.738231
Name: Ai_drive, dtype: float64

In [1016]:
kznsal["Ai_drive"].describe(percentiles=[.9,0.95, 0.98, 0.99, 0.999])

count    14094.000000
mean         0.054672
std          0.576564
min          0.000000
50%          0.000000
90%          0.075258
95%          0.174195
98%          0.534855
99%          1.113023
99.9%        3.695485
max         32.738231
Name: Ai_drive, dtype: float64

In [ ]:
# Gauteng Walk Analysis 

In [859]:
# gauwalk=ox.load_graphml('../../networks/network_gauteng_walk.graphml')
gauwalk = ox.project_graph(gauwalk, to_crs="EPSG:32735")
gauwalk.graph["crs"]

<Projected CRS: EPSG:32735>
Name: WGS 84 / UTM zone 35S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 24°E and 30°E, southern hemisphere between 80°S and equator, onshore and offshore. Botswana. Burundi. Democratic Republic of the Congo (Zaire). Rwanda. South Africa. Tanzania. Uganda. Zambia. Zimbabwe.
- bounds: (24.0, -80.0, 30.0, 0.0)
Coordinate Operation:
- name: UTM zone 35S
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [1017]:
gausal = gausal.set_geometry(salpoint["geometry"])

gausal = gausal.to_crs(gauwalk.graph["crs"])
gaupharm = gaupharm.to_crs(gauwalk.graph["crs"])

In [ ]:
# GRAPH TRUNCATION
(maxy, miny, maxx, minx) = gausal.total_bounds 
bbox = (maxy, miny, maxx, minx)

gauwalk = ox.truncate.truncate_graph_bbox(gauwalk, bbox)

In [ ]:
# NODE ASSIGNMENT
gausal["node"] = ox.nearest_nodes(
    gauwalk,
    gausal.geometry.x,
    gausal.geometry.y
)

pharm_gdf = gaupharm.copy()
# pharm_gdf = pharm_gdf[
#     pharm_gdf.geometry.x.notna() &
#     pharm_gdf.geometry.y.notna()
# ]

pharm_gdf["node"] = ox.nearest_nodes(
    gauwalk,
    pharm_gdf.geometry.x,
    pharm_gdf.geometry.y
)

In [1019]:
pop_dict = gausal.set_index("node")["sal2023_est"].to_dict()

Rj_list = []

for idx, pharm in tqdm(pharm_gdf.iterrows(), total=len(pharm_gdf)):

    lengths = nx.single_source_dijkstra_path_length(
        gauwalk,
        pharm["node"],
        cutoff=2000,
        weight="length"
    )

    weighted_pop = sum(
        pop_dict.get(node, 0) * decay(dist)
        for node, dist in lengths.items()
    )


    Rj = 1 / (weighted_pop/1000) if weighted_pop > 0 else 0
    Rj_list.append(Rj)

pharm_gdf["Rj"] = Rj_list

pharm_dict = pharm_gdf.groupby("node")["Rj"].sum().to_dict()

Ai_list = []

for idx, pop in tqdm(gausal.iterrows(), total=len(gausal)):

    lengths = nx.single_source_dijkstra_path_length(
        gauwalk,
        pop["node"],
        cutoff=2000,
        weight="length"
    )

    Ai = sum(
        pharm_dict.get(node, 0) * decay(dist)
        for node, dist in lengths.items()
    )

    Ai_list.append(Ai)

gausal["Ai_walk"] = Ai_list

100%|███████████████████████████████████████████████████████████████████████████| 17046/17046 [00:31<00:00, 532.79it/s]


In [1020]:
weighted_pop

np.float64(26991.30082920599)

In [1023]:
gausal["Ai_walk"].describe() 

count    17046.000000
mean         0.052544
std          0.111410
min          0.000000
25%          0.000000
50%          0.000000
75%          0.057791
max          3.290218
Name: Ai_walk, dtype: float64

In [ ]:
gausal["Ai_walk"].describe(percentiles=[.9,0.95, 0.98, 0.99, 0.999])

count    17046.000000
mean         0.052544
std          0.111410
min          0.000000
50%          0.000000
90%          0.173590
95%          0.259724
98%          0.368210
99%          0.452637
99.9%        1.163835
max          3.290218
Name: Ai_walk, dtype: float64

In [ ]:
# Gauteng Road Analysis 
# Drive Catchment: 8km (
# Pharmacy Population Demand Catchment: 8km

# Pharmacy Supply: 10000/catchment pop (we must assume equal supply accross pharamcies due to limited data)
#Here we assume double pharmacy supply due to existing literature that Gauteng has the most medical resources within SA and a larger demand 

# Pharmacy Catchment Population Distribution
# mean      380.677203
# std       268.025377
# min         0.000000
# 25%       229.249150
# 50%       322.719567
# 75%       456.931665
# max      2282.328768

In [44]:
#gaudrive=ox.load_graphml('../../networks/network_gauteng_drive.graphml')
gaudrive = ox.project_graph(gaudrive)
gaudrive.graph["crs"]

<Projected CRS: EPSG:32735>
Name: WGS 84 / UTM zone 35S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 24°E and 30°E, southern hemisphere between 80°S and equator, onshore and offshore. Botswana. Burundi. Democratic Republic of the Congo (Zaire). Rwanda. South Africa. Tanzania. Uganda. Zambia. Zimbabwe.
- bounds: (24.0, -80.0, 30.0, 0.0)
Coordinate Operation:
- name: UTM zone 35S
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
gausal = gausal.to_crs(gaudrive.graph["crs"])
gaupharm = gaupharm.to_crs(gaudrive.graph["crs"])

In [ ]:
# GRAPH TRUNCATION
(maxy, miny, maxx, minx) = gausal.total_bounds 
bbox = (maxy, miny, maxx, minx)

gaudrive = ox.truncate.truncate_graph_bbox(gaudrive, bbox)

In [ ]:
# NODE ASSIGNMENT
gausal["node"] = ox.nearest_nodes(
    gaudrive,
    gausal.geometry.x,
    gausal.geometry.y
)

pharm_gdf["node"] = ox.nearest_nodes(
    gaudrive,
    pharm_gdf.geometry.x,
    pharm_gdf.geometry.y
)

In [1028]:
pop_dict = gausal.set_index("node")["sal2023_est"].to_dict()

Rj_list = []

for idx, pharm in tqdm(pharm_gdf.iterrows(), total=len(pharm_gdf)):

    lengths = nx.single_source_dijkstra_path_length(
        gaudrive,
        pharm["node"],
        cutoff=5000,
        weight="length"
    )

    weighted_pop = sum(
        pop_dict.get(node, 0) * decay(dist)
        for node, dist in lengths.items()
    )


    Rj = 1 / (weighted_pop/1000) if weighted_pop > 0 else 0
    Rj_list.append(Rj)

pharm_gdf["Rj"] = Rj_list

pharm_dict = pharm_gdf.groupby("node")["Rj"].sum().to_dict()

Ai_list = []

for idx, pop in tqdm(gausal.iterrows(), total=len(gausal)):

    lengths = nx.single_source_dijkstra_path_length(
        gaudrive,
        pop["node"],
        cutoff=5000,
        weight="length"
    )

    Ai = sum(
        pharm_dict.get(node, 0) * decay(dist)
        for node, dist in lengths.items()
    )

    Ai_list.append(Ai)

gausal["Ai_drive"] = Ai_list

100%|███████████████████████████████████████████████████████████████████████████| 17046/17046 [01:45<00:00, 162.11it/s]


In [1029]:
weighted_pop

np.float64(125846.41377919918)

In [1030]:
gausal["Ai_drive"].describe() 

count    17046.000000
mean         0.103102
std          4.059240
min          0.000000
25%          0.000000
50%          0.020096
75%          0.084213
max        527.832622
Name: Ai_drive, dtype: float64

In [1031]:
gausal["Ai_drive"].describe(percentiles=[.9,0.95, 0.98, 0.99, 0.999])

count    17046.000000
mean         0.103102
std          4.059240
min          0.000000
50%          0.020096
90%          0.158197
95%          0.222176
98%          0.320756
99%          0.875774
99.9%        1.931852
max        527.832622
Name: Ai_drive, dtype: float64

In [ ]:
gausal["Ai_drive"].hist()

In [ ]:
#allaccess['area_km2'].describe()

In [ ]:
kznsal["walk_rank"] = 0.0
mask = kznsal["Ai_walk"] > 0
kznsal.loc[mask, "walk_rank"] = kznsal.loc[mask, "Ai_walk"].rank(pct=True)

kznsal["drive_rank"] = 0.0
mask = kznsal["Ai_drive"] > 0
kznsal.loc[mask, "drive_rank"] = kznsal.loc[mask, "Ai_drive"].rank(pct=True)

gausal["walk_rank"] = 0.0
mask = gausal["Ai_walk"] > 0
gausal.loc[mask, "walk_rank"] = gausal.loc[mask, "Ai_walk"].rank(pct=True)

gausal["drive_rank"] = 0.0
mask = gausal["Ai_drive"] > 0
gausal.loc[mask, "drive_rank"] = gausal.loc[mask, "Ai_drive"].rank(pct=True)

In [1052]:
kznsal["walk_log"] =  np.log1p(kznsal["Ai_walk"])
kznsal["drive_log"] = np.log1p(kznsal["Ai_drive"])
gausal["walk_log"] =  np.log1p(gausal["Ai_walk"])
gausal["drive_log"] =  np.log1p(gausal["Ai_drive"])
allaccess = pd.concat([kznsal, gausal], ignore_index=True)
allaccess = allaccess.drop(columns='geometry')

allaccess = salpoly.merge(allaccess, on='EA_CODE', how='left')
cols = ['walk_log', 'drive_log',"walk_rank", "drive_rank" ]

allaccess[cols] = allaccess[cols].fillna(0)

C:\Users\kalmanj\AppData\Local\Temp\ipykernel_36976\437139378.py:8: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  allaccess = salpoly.merge(allaccess, on='EA_CODE', how='left')


In [1053]:
for c in cols:
    allaccess[c] = pd.to_numeric(allaccess[c], errors="coerce")

In [1054]:
allaccess['geometry'] = allaccess.geometry.to_wkt()
allaccess.to_csv(
    "../../new_allaccess.csv",
    index=False,
    float_format="%.6f",   # keeps decimals like 0.923456
    na_rep=""              # no 'nan' strings
)

C:\Users\kalmanj\AppData\Local\Temp\ipykernel_36976\2280266202.py:1: UserWarning: Geometry column does not contain geometry.
  allaccess['geometry'] = allaccess.geometry.to_wkt()


In [ ]:
### MAPS

In [883]:
type(allaccess)

geopandas.geodataframe.GeoDataFrame

In [3]:
allaccess=pd.read_csv('../../new_allaccess.csv')

C:\Users\kalmanj\AppData\Local\Temp\ipykernel_19396\2693392590.py:1: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  allaccess=pd.read_csv('../../new_allaccess.csv')


In [7]:
salpoly = salpoly.to_crs(epsg=3857)
allaccess = allaccess.drop(columns=["geometry"], errors="ignore").merge(
    salpoly[["EA_CODE", "geometry"]],
    on="EA_CODE",
    how="left",
  )
import geopandas as gpd

allaccess = gpd.GeoDataFrame(
    allaccess,
    geometry="geometry",
    crs=salpoly.crs  # or "EPSG:3857" if already projected
)

In [1059]:
pharm_with_sal = pharm_with_sal.to_crs(allaccess.crs)

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib as mpl

# cols = ["walk_log", "drive_log"]

# bin_edges = [0, 0.025, 0.1, 1]

# fig, axes = plt.subplots(1, 2, figsize=(14,8))

# cmap = mpl.cm.get_cmap("RdYlGn", 3)
# for i, col in enumerate(cols):

#     allaccess.plot(
#         column=col,
#         cmap=cmap,
#         scheme='UserDefined',
#         classification_kwds={'bins': bin_edges},
#         legend=False,
#         ax=axes[i],
#         edgecolor="none"
#     )

#     axes[i].set_title(f"{col.capitalize()} Accessibility")
#     axes[i].axis('off')

# # shared colorbar
# norm = mpl.colors.BoundaryNorm(bin_edges, ncolors=cmap.N)
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])

# cbar = fig.colorbar(sm, ax=axes, fraction=0.03, pad=0.02)
# cbar.set_label("Accessibility Score")

# plt.suptitle("Walking vs Driving Accessibility")

# plt.savefig("images/walk_vs_drive.png", dpi=150, bbox_inches='tight')
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

bin_edges = [0, 0.048, 0.1, 3]

fig, ax = plt.subplots(figsize=(8,8))
cmap = mpl.cm.get_cmap("RdYlGn", 3)


norm = mpl.colors.BoundaryNorm(bin_edges, ncolors=cmap.N)

# polygon layer (walking)
allaccess.plot(
    column='walk_log',
    cmap=cmap,
    scheme='UserDefined',
    classification_kwds={'bins': bin_edges},
    ax=ax,
    edgecolor="none"
)

# pharmacy points
pharm_with_sal.plot(
    ax=ax,
    color='blue',
    markersize=3,
    alpha=0.7
)

# Johannesburg zoom
ax.set_xlim(3080000, 3150000)
ax.set_ylim(-3050000, -2970000)
scalebar = ScaleBar(
    dx=1,              # 1 unit = 1 meter (because of EPSG:3857)
    units="m",
    dimension="si-length",
    location='upper left'
)      
# colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Walking Accessibility")
ax.add_artist(scalebar)
ax.set_title("Walking Accessibility (Johannesburg)")
ax.axis('off')
plt.savefig("images/joburg_walk.png", dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, ax = plt.subplots(figsize=(8,8))

bin_edges = [0, 0.017, 0.05, 3]

cmap = mpl.cm.get_cmap("RdYlGn", 3)
norm = mpl.colors.BoundaryNorm(bin_edges, ncolors=cmap.N)

# polygons
allaccess.plot(
    column='walk_log',
    cmap=cmap,
    norm=norm,   
    edgecolor="none",
    ax=ax
)

# make sure CRS matches
pharm_with_sal = pharm_with_sal.to_crs(allaccess.crs)

# pharmacies
pharm_with_sal.plot(
    ax=ax,
    color='blue',
    markersize=3,
    alpha=0.7
)
scalebar = ScaleBar(
    dx=1,              # 1 unit = 1 meter (because of EPSG:3857)
    units="m",
    dimension="si-length",
    location='lower right'
)  
ax.add_artist(scalebar)

# Durban zoom
ax.set_xlim(3434000, 3480000)
ax.set_ylim(-3506000, -3448000)

# colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Accessibility Score")

ax.set_title("Durban Walking Accessibility")
ax.axis('off')
plt.savefig("images/durban_walk.png", dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

bin_edges = [0, 0.025,.05, 3]
cmap = mpl.cm.RdYlGn

norm = mpl.colors.BoundaryNorm(bin_edges, cmap.N)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
cities = {
    "Johannesburg": (3050000, 3170000, -3050000, -2950000),
    "Durban": (3434000, 3480000, -3506000, -3448000)
}
 
ax.add_artist(scalebar)
for i, (name, (xmin, xmax, ymin, ymax)) in enumerate(cities.items()):

    gdf = allaccess.cx[xmin:xmax, ymin:ymax]

    gdf.plot(
        column='walk_log',
        cmap=cmap,
        norm=norm,         
        ax=axes[i],
        edgecolor='none'
    )

    # pharm_with_sal.cx[xmin:xmax, ymin:ymax].plot(
    #     ax=axes[i],
    #     color='#3B0C87',
    #     markersize= 1,
    #     zorder=10
    # )

    axes[i].set_title(name)
    axes[i].axis('off')
scalebar = ScaleBar(
    dx=1,              # 1 unit = 1 meter (because of EPSG:3857)
    units="m",
    dimension="si-length",
    location='lower right'
) 
ax.add_artist(scalebar)

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=axes, fraction=0.03, pad=0.02)
cbar.set_label("Accessibility Score")
plt.suptitle("Walking Accessibility: Johannesburg vs Durban")
#plt.savefig("joburgandDurban_walking.png", dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

bin_edges = [0, 0.25, 0.5, 0.75,1]
cmap = mpl.cm.RdYlGn

norm = mpl.colors.BoundaryNorm(bin_edges, cmap.N)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

cities = {
    "Johannesburg": (27.7, 28.3, -26.4, -25.8),
    "Durban": (30.85, 31.25, -30.05, -29.65)
}

for i, (name, (xmin, xmax, ymin, ymax)) in enumerate(cities.items()):

    gdf = allaccess.cx[xmin:xmax, ymin:ymax]

    gdf.plot(
        column='drive_score',
        cmap=cmap,
        norm=norm,         
        ax=axes[i],
        edgecolor='none'
    )

    pharm_with_sal.cx[xmin:xmax, ymin:ymax].plot(
        ax=axes[i],
        color='blue',
        markersize= 1,
        zorder=10
    )

    axes[i].set_title(name)
    axes[i].axis('off')

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=axes, fraction=0.03, pad=0.02)
cbar.set_label("Accessibility Score")

plt.suptitle("Driving Accessibility: Johannesburg vs Durban")
#plt.savefig("joburgandDurban_driving.png", dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
pip install matplotlib-scalebar

In [ ]:
import matplotlib.pyplot as plt
highlight_codes = [
56010074,
54610011,
54610001,
54610002,
56010140,
56010136,
56010131,
56010138,
56010134,
56010137,

]  

fig, ax = plt.subplots(figsize=(8,8))

gdf_prov = allaccess[allaccess['PR_NAME'] == 'KwaZulu-Natal']

subset = gdf_prov[gdf_prov['EA_CODE'].isin(highlight_codes)]

# base
gdf_prov.plot(ax=ax, 
              color='darkgrey',
              edgecolor='lightgrey'   )

# highlight
subset.plot(ax=ax, color='#DE3831',  
            linewidth=0.6,
           edgecolor='black')
#pharmacy points
pharm_with_sal.plot(
    ax=ax,
    color='#001147',
    markersize=4,
    alpha=0.8,
    edgecolor="#2E5FFF"
)
scalebar = ScaleBar(
    dx=1,              # 1 unit = 1 meter (because of EPSG:3857)
    units="m",
    dimension="si-length",
    location='lower right'
)

# after polygons + pharmacies
# Durban
ax.scatter(3453460, -3488260, color='black', s=25, zorder=20)
ax.text(3453460 +5000, -3488260, "Durban", fontsize=10, ha='left')

ax.scatter(3446500, -3421000, color='black', s=25, zorder=20)
ax.text(3446500, -3421000-5000 , "Ndwedwe", fontsize=10)

ax.scatter(3409300, -3516000, color='black', s=25, zorder=20)
ax.text(3409300 , -3516000 - 6000, "Vulamehlo", fontsize=10)

# # Msinga
# ax.scatter(3391030, -3337800, color='black', s=25, zorder=20)
# ax.text(3391030 - 5000, -3337800 + 5000, "Msinga", fontsize=10, ha='left')

# 🔥 AUTO ZOOM
minx, miny, maxx, maxy = subset.total_bounds

# optional padding a
pad_x = (maxx - minx) *.5
pad_y = (maxy - miny) *.25

ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)
ax.add_artist(scalebar)
ax.set_title("Bottom 10 Walking Accessibility Score - KZN")
ax.axis('off')
plt.savefig("images/bottomwalkkzn.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
from matplotlib_scalebar.scalebar import ScaleBar
import matplotlib.pyplot as plt

highlight_codes = [
52910145,
52910137,
59210304,
58610076,
52910134,
57910165,
57610118,
59210227,
59910273,
57610147,


]

fig, ax = plt.subplots(figsize=(8,8))

gdf_prov = allaccess[allaccess['PR_NAME'] == 'KwaZulu-Natal']
subset = gdf_prov[gdf_prov['EA_CODE'].isin(highlight_codes)]

# --- PLOT BASE ---
gdf_prov.plot(ax=ax, color='darkgrey', edgecolor='lightgrey')
subset.plot(ax=ax, color='#DE3831', linewidth=0.6, edgecolor='#DE3831')

# pharm_with_sal.plot(
#     ax=ax,
#     color='#001147',
#     markersize=.7,
#     alpha=0.5,
#     edgecolor="#2E5FFF"
# )

# --- DURBAN POINT ---
durban_x, durban_y = 3453460, -3488260

ax.scatter(durban_x, durban_y, color='black', s=15, zorder=20)
ax.text(durban_x - 3000, durban_y-15000, "Durban", fontsize=10)
ax.scatter(3461500, -3442000-10000, color='black', s=15, zorder=20)
ax.text(3461500 , -3442000, "Umhlanga", fontsize=10)
# ax.scatter(3376000, -3430000+15000, color='black', s=15, zorder=20)
# ax.text(3376000 - 30000, -3430000, "Msunduzi", fontsize=10)
# ax.scatter(3346000, -3178000, color='black', s=15, zorder=20)
# ax.text(3346000 - 5000, -3178000-15000, "Newcastle", fontsize=10)
# ax.scatter(3390700, -3552000-10000, color='black', s=15, zorder=20)
# ax.text(3390700 , -3552000, "Hibiscus Coast", fontsize=10)
# ax.scatter(3439000, -3239000, color='black', s=15, zorder=20)
# ax.text(3439000 - 50000, -3239000-30000, "KwaDukuza", fontsize=10)
ax.scatter(3562000, -3336000, color='black', s=15, zorder=20)
ax.text(3562000 - 30000, -3336000+10000, "Empangeni", fontsize=10)

ax.scatter(3391030, -3337800, color='black', s=25, zorder=20)
ax.text(3391030 - 5000, -3337800 - 15000, "Msinga", fontsize=10, ha='left')
minx, miny, maxx, maxy = subset.total_bounds
# include Durban in bounds
minx = min(minx, durban_x)
maxx = max(maxx, durban_x)
miny = min(miny, durban_y)
maxy = max(maxy, durban_y)

# padding
pad_x = (maxx - minx)*.3
pad_y = (maxy - miny)*.3 

ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)

# --- SCALE BAR ---
scalebar = ScaleBar(dx=1, units="m", location='lower right')
ax.add_artist(scalebar)

# --- TITLES ---
ax.set_title("Bottom 10 Drive Accessibility Score - KZN", fontsize=12)

fig.text(
    0.5, 0.88,
    "For areas > 100 population",
    ha='center',
    fontsize=9,
    color='gray'
)

ax.axis('off')

plt.savefig("images/bottomdrivekzn.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
highlight_codes = [
76310653,
79911566,
79914587,
76310630,
76410007,
79914588,
76310565,
76610048,
76610029,
79910975,

]  

fig, ax = plt.subplots(figsize=(8,8))

gdf_prov = allaccess[allaccess['PR_NAME'] == 'Gauteng']

subset = gdf_prov[gdf_prov['EA_CODE'].isin(highlight_codes)]

# base
gdf_prov.plot(ax=ax, 
              color='darkgrey',
              edgecolor='lightgrey'   )

# highlight
subset.plot(ax=ax, color='#DE3831',  
            linewidth=0.6,
           edgecolor='#DE3831')
#pharmacy points
pharm_with_sal.plot(
    ax=ax,
    color='#001147',
    markersize=.75,
    alpha=0.25,
    edgecolor="#2E5FFF"
)
scalebar = ScaleBar(
    dx=1,              # 1 unit = 1 meter (because of EPSG:3857)
    units="m",
    dimension="si-length",
    location='lower right'
)

# after polygons + pharmacies
# 
ax.scatter(3118870, -3005460, color='black', s=10, zorder=20)
ax.text(3118870 -15000, -3005460+7000, "Johannesburg", fontsize=10, ha='left')
ax.scatter(3142400, -2952500, color='black', s=10, zorder=20)
ax.text(3142400 , -2952500 + 5000, "Pretoria", fontsize=10)
# ax.scatter(3049000, -3024000, color='black', s=10, zorder=20)
# ax.text(3049000 - 20000, -3024000+7000  , "Merafong City", fontsize=10)




# 🔥 AUTO ZOOM
minx, miny, maxx, maxy = subset.total_bounds

# optional padding 
pad_x = (maxx - minx)*0.3
pad_y = (maxy - miny)*0.3

ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)
ax.add_artist(scalebar)

ax.set_title("Bottom 10 Driving Accessibility Score - Gauteng", fontsize=12,)

# subtitle
fig.text(
    0.5,0.85,
    "For areas > 100 population",
    ha='center',
    fontsize=9,
    color='gray'
)

ax.axis('off')
plt.savefig("images/bottomdrivegauteng.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
highlight_codes = [
79712324,
79712083,
76011251,
79710301,
79715028,
76010707,
76010705,
76010586,
79714933,
79710300,
]  

fig, ax = plt.subplots(figsize=(8,8))

gdf_prov = allaccess[allaccess['PR_NAME'] == 'Gauteng']

subset = gdf_prov[gdf_prov['EA_CODE'].isin(highlight_codes)]

# base
gdf_prov.plot(ax=ax, 
              color='darkgrey',
              edgecolor='lightgrey'   )

# highlight
subset.plot(ax=ax, color='#DE3831',  
            linewidth=0.6,
           edgecolor='#DE3831')
#pharmacy points
pharm_with_sal.plot(
    ax=ax,
    color='#001147',
    markersize=1,
    alpha=0.8,
    edgecolor="#2E5FFF"
)
Scalebar = ScaleBar(
    dx=1,              # 1 unit = 1 meter (because of EPSG:3857)
    units="m",
    dimension="si-length",
    location='lower right'
)

# after polygons + pharmacies
# 
ax.scatter(3118870, -3005460, color='black', s=25, zorder=20)
ax.text(3118870 -32000, -3005460+5000, "Johannesburg", fontsize=10, ha='left')

# 
# ax.scatter(3089800, -3099600, color='black', s=25, zorder=20)
ax.text(3089800 - 5000, -3099600 + 6000, "Marlbank River", fontsize=10, ha='left')
# ax.scatter(3141600, -3112400, color='black', s=25, zorder=20)
ax.text(3141600 + 5000, -3112400 , "Vaal Marina", fontsize=10, ha='left')
# ax.scatter(3113800, -3015400, color='black', s=25, zorder=20)
# ax.text(3112100-30000 , -3019900-10000 , "Meredale/Mondeor", fontsize=10, ha='left')
ax.scatter(3142400, -2952500, color='black', s=25, zorder=20)
ax.text(3142400 , -2952500 + 5000, "Pretoria", fontsize=10)
ax.scatter(3049000, -3024000, color='black', s=25, zorder=20)
ax.text(3049000 - 7000, -3024000+7000  , "Merafong City", fontsize=10)


# 🔥 AUTO ZOOM
minx, miny, maxx, maxy = subset.total_bounds

# optional padding 
pad_x = (maxx - minx) *.3
pad_y = (maxy - miny) *0.2

ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)
ax.set_title("Top 10 Walking Accessibility Score - Gauteng")
ax.axis('off')
plt.savefig("images/bottomwalkgauteng.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# ox.settings.use_cache = True

# top10_walk = [79913191,
# 76010001,
# 76110002,
# 79911565,
# 76110001,
# 50610302,
# 50610303,
# 76310678,
# 76310681,
# 76310680,
# ]
# bottom10_walk = [59914267,
# 59914272,
# 59913977,
# 59914251,
# 59914249,
# 59913982,
# 59914927,
# 59914184,
# 59914187,
# 59914197,
# ]
# top10_drive = [79913191,
# 79814126,
# 79814112,
# 79814128,
# 79814136,
# 79815745,
# 79814137,
# 79815736,
# 79814114,
# 79814127,
# ]
# bottom10_drive = [59914140,
# 59915043,
# 59914126,
# 59915036,
# 59914137,
# 59915050,
# 59914147,
# 59915054,
# 59915057,
# 59914138]

# import geopandas as gpd
# import folium

# def classify(row):
#     if row['EA_CODE'] in top10_walk:
#         return 'Top Walk'
#     elif row['EA_CODE'] in bottom10_walk:
#         return 'Bottom Walk'
#     elif row['EA_CODE'] in top10_drive:
#         return 'Top Drive'
#     elif row['EA_CODE'] in bottom10_drive:
#         return 'Bottom Drive'
#     else:
#         return 'Other'
        
# gdf= allaccess.copy()
# gdf['category'] = gdf.apply(classify, axis=1)

# # ----------------------------
# # 3. Colors
# # ----------------------------
# color_dict = {
#     'Top Walk': '#1a9641',     # green
#     'Bottom Walk': '#d7191c',  # red
#     'Top Drive': '#2b83ba',    # blue
#     'Bottom Drive': '#fdae61', # orange
#     'Other': '#d3d3d3'         # grey
# }

# # ----------------------------
# # 4. Convert to WGS84 (REQUIRED for folium)
# # ----------------------------
# gdf = gdf.to_crs(epsg=4326)

# # ----------------------------
# # 5. Map center using bounds (NO centroid)
# # ----------------------------
# minx, miny, maxx, maxy = gdf.total_bounds

# m = folium.Map(
#     location=[(miny + maxy) / 2, (minx + maxx) / 2],
#     zoom_start=11
# )

# # ----------------------------
# # 6. Fix data types for folium
# # ----------------------------
# fields = [col for col in gdf.columns if col != 'geometry']
# gdf[fields] = gdf[fields].astype(str)

# # ----------------------------
# # 7. Add layers by category
# # ----------------------------
# for cat in gdf['category'].unique():
#     subset = gdf[gdf['category'] == cat]

#     folium.GeoJson(
#         subset,
#         name=cat,
#         style_function=lambda feature, cat=cat: {
#             'fillColor': color_dict[cat],
#             'color': 'lightgray',
#             'weight': 1,
#             'fillOpacity': 0.7
#         },
#         tooltip=folium.GeoJsonTooltip(
#             fields=['EA_CODE', 'category'],  # add more if needed
#             aliases=['Area:', 'Group:']
#         )
#     ).add_to(m)

# # ----------------------------
# # 8. Add layer control
# # ----------------------------
# folium.LayerControl().add_to(m)

# # ----------------------------
# # 9. OPTIONAL: auto-zoom to your data (much better UX)
# # ----------------------------
# m.fit_bounds([[miny, minx], [maxy, maxx]])

# # ----------------------------
# # 10. Save
# # ----------------------------
# m.save("accessibility_test.html")

In [976]:
allaccess[["walk_q", "walk_score"]].describe()

,walk_q,walk_score
count,"38,380.0000","38,380.0000"
mean,0.2158,0.1990
std,0.5596,0.4736
min,0.0000,0.0000
25%,0.0000,0.0000
50%,0.0000,0.0000
75%,0.0000,0.0000
max,2.0000,4.1529


In [1060]:
allaccess = allaccess.to_crs("EPSG:3857")

In [ ]:
from matplotlib_scalebar.scalebar import ScaleBar

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- QUANTILES ---
allaccess["walk_q"] = 0

mask = allaccess["walk_log"] > 0
allaccess.loc[mask, "walk_q"] = pd.qcut(
    allaccess.loc[mask, "walk_log"],
    3,
    labels=[0,1,2]
)

allaccess["drive_q"] = 0

mask = allaccess["drive_log"] > 0
allaccess.loc[mask, "drive_q"] = pd.qcut(
    allaccess.loc[mask, "drive_log"],
    3,
    labels=[0,1,2]
)

# --- COLORS ---
biv_colors = {
    (0,0): "#E7E7E7",

    (0,1): "#FFF2CC",
    (0,2): "#FFC000",

    (1,0): "#C6EFCE",
    (1,1): "#92D18E",
    (1,2): "#92D050",

    (2,0): "#00784B",
    (2,1): "#00A69B",
    (2,2): "#005096"  # softened blue anchor
}

# --- ASSIGN COLORS ---
allaccess["biv_color"] = allaccess.apply(
    lambda r: biv_colors[(int(r["walk_q"]), int(r["drive_q"]))],
    axis=1
)

# =========================
# PLOT (ONLY ONCE)
# =========================
fig, ax = plt.subplots(figsize=(10,10))

allaccess.plot(
    color=allaccess["biv_color"],
    ax=ax,
    linewidth=0.1,
    edgecolor="white"
)
ax.scatter(3118870, -3005460, color='black', s=5, zorder=20)
ax.text(3118870 -85000, -3005460+20000, "Johannesburg", fontsize=9, ha='left')
ax.scatter(3142400, -2952500, color='black', s=5, zorder=20)
ax.text(3142400-30000 , -2952500 + 20000, "Pretoria", fontsize=9)

ax.scatter(3453460, -3488260, color='black', s=5, zorder=20)
ax.text(3453460 +5000, -3488260, "Durban", fontsize=9, ha='left')
# ax.scatter(3391030, -3337800, color='black', s=5, zorder=20)
# ax.text(3391030 - 5000, -3337800 - 15000, "Msinga", fontsize=9, ha='left')
ax.scatter(3449000, -3465000, color='black', s=5, zorder=20)
ax.text(3449000 - 5000, -3465000 + 15000, "KwaMashu", fontsize=9, ha='left')

ax.set_title("Bivariate Accessibility: Walking vs Driving Accessibility Scores", fontsize=14)
ax.axis("off")

# =========================
# LEGEND (SAME FIGURE)
# =========================
import matplotlib.patches as mpatches

import matplotlib.patches as mpatches
import pandas as pd

# counts (make sure defined)
counts = pd.crosstab(allaccess["walk_q"], allaccess["drive_q"])

legend_ax = fig.add_axes([0.75, 0.15, 0.2, 0.2])
legend_ax.axis("off")

import matplotlib.patches as mpatches

for i in range(3):          # walk (0=low bottom → 2=high top)
    for j in range(3):      # drive (0=low left → 2=high right)

        color = biv_colors[(i, j)]

        rect = mpatches.Rectangle(
            (j, i),   # x = drive, y = walk
            1, 1,
            facecolor=color,
            edgecolor="white"
        )
        legend_ax.add_patch(rect)

        value = counts.loc[i, j] if (i in counts.index and j in counts.columns) else 0

        legend_ax.text(
            j + 0.5,
            i + 0.5,
            f"{value}",
            ha="center",
            va="center",
            fontsize=8,
            color="black"
        )
scalebar = ScaleBar(
    dx=1,              # 1 unit = 1 meter (because of EPSG:3857)
    units="m",
    dimension="si-length",
    location='lower left'
)      
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

axins = inset_axes(ax, width="30%", height="30%", bbox_to_anchor=(0, 0, .8, 1), loc="upper right",  bbox_transform=ax.transAxes)
axins2 = inset_axes(
    ax,
    width="30%",
    height="30%",
    loc="lower left",
    bbox_to_anchor=(0.01, 0.05, 1, 1),
    bbox_transform=ax.transAxes
)

# plot SAME data into inset
allaccess.plot(
    color=allaccess["biv_color"],
    ax=axins,
    linewidth=0.1,
    edgecolor="white"
)
allaccess.plot(
    color=allaccess["biv_color"],
    ax=axins2,
    linewidth=0.1,
    edgecolor="white"
)

# --- ZOOM TO GAUTENG ---
axins.set_xlim(3090000, 3170000)
axins.set_ylim(-3035000, -2930000)
axins.scatter(3128000, -2983000, color='black', s=5, zorder=20)
axins.text(3128000 + 5000, -2983000 - 5000, "Olievenhoutbosch", fontsize=9, ha='left')


# remove axis clutter
axins.set_xticks([])
axins.set_yticks([])
axins.set_title("Johannesburg / Pretoria", fontsize=8)


# --- ZOOM TO DURBAN ---
axins2.set_xlim(3428000, 3492000)
axins2.set_ylim(-3511000, -3439000)

axins2.scatter(3449000, -3465000, color='black', s=5, zorder=20)
axins2.text(3449000 + 5000, -3465000 + 5000, "KwaMashu", fontsize=8, ha='left')

# clean up
axins2.set_xticks([])
axins2.set_yticks([])
axins2.set_title("Durban", fontsize=8)
ax.add_artist(scalebar)
legend_ax.set_xlim(0, 3)
legend_ax.set_ylim(0, 3)
# Axis labels
legend_ax.text(1.5, -0.4, "Driving Access →", ha="center", fontsize=9)
legend_ax.text(-0.4, 1.5, "Walking Access →", va="center", rotation=90, fontsize=9)
plt.savefig("images/bivariate.png", dpi=300, bbox_inches='tight')

plt.show()